In [0]:
catalog = "workspace"
schema = "certification_pipeline_lab"

volume_root = (
    f"/Volumes/{catalog}/{schema}/raw_files"
)

orders_path = f"{volume_root}/orders"
status_path = f"{volume_root}/status"
customers_cdc_path = f"{volume_root}/customers_cdc"

In [0]:
new_orders = [
    (
        3001,
        "2026-09-06T10:00:00",
        4,
        "IT",
        199.99,
        "Y"
    )
]

new_orders_df = spark.createDataFrame(
    new_orders,
    [
        "order_id",
        "order_timestamp",
        "customer_id",
        "country_code",
        "amount",
        "notifications"
    ]
)

(
    new_orders_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(
        f"{orders_path}/batch_003"
    )
)

In [0]:
new_statuses = [
    (
        3001,
        "DELIVERED",
        "2026-09-06T15:30:00"
    )
]

new_statuses_df = spark.createDataFrame(
    new_statuses,
    [
        "order_id",
        "status",
        "status_timestamp"
    ]
)

(
    new_statuses_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(
        f"{status_path}/batch_004"
    )
)

In [0]:
%sql
UPDATE workspace.certification_pipeline_lab.ref_countries

SET capital = 'PARIS_V2'

WHERE country_code = 'FR';

In [0]:
%sql
SELECT
    'STREAMING_TABLE' AS dataset,
    order_id,
    capital

FROM workspace.certification_pipeline_lab.orders_enriched_silver

WHERE country_code = 'FR'

UNION ALL

SELECT
    'MATERIALIZED_VIEW' AS dataset,
    order_id,
    capital

FROM workspace.certification_pipeline_lab.full_order_info_gold

WHERE country_code = 'FR'

ORDER BY order_id, dataset;

In [0]:
new_fr_order = [
    (
        3002,
        "2026-09-07T10:00:00",
        1,
        "FR",
        80.00,
        "Y"
    )
]

new_fr_order_df = spark.createDataFrame(
    new_fr_order,
    [
        "order_id",
        "order_timestamp",
        "customer_id",
        "country_code",
        "amount",
        "notifications"
    ]
)

(
    new_fr_order_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(
        f"{orders_path}/batch_004"
    )
)